# 4-2절 연습 문제 풀이

이 노트북은 4-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch04/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# MNIST 데이터셋 준비 (내려받기 경로는 저장소의 downloads 디렉터리)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

DATA_ROOT = '../../downloads'

def mnist_loaders(batch_size=64, transform=None, valid_ratio=0.2):
    transform = transform or transforms.ToTensor()
    full = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
    n_valid = int(len(full) * valid_ratio)
    g = torch.Generator().manual_seed(SEED)
    train_set, valid_set = random_split(full, [len(full) - n_valid, n_valid], generator=g)
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(valid_set, batch_size=batch_size),
            DataLoader(test_set, batch_size=batch_size))

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            loss = criterion(out, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (out.argmax(dim=1) == labels).sum().item()
            n += labels.size(0)
    return total_loss / n, correct / n * 100

# 3장 회오리 데이터 (4-7에서 사용)
import csv

def load_spiral(path='../../data/ch3_spiral_data.csv'):
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    X = torch.tensor([[float(r['x1']), float(r['x2'])] for r in rows])
    Y = torch.tensor([int(float(r['label'])) for r in rows])
    return X, Y

## 연습 4-5

[코드 4-16]은 배치 학습 함수, 배치 검증 함수와 조기 종료 적용 구현이 생략되어 있다. 빠진 부분을 완성한 후 조기 종료 방식으로 모델을 학습해 보자. 참고로 깃허브 노트북 예제에는 완성된 학습 함수가 게시되어 있다. 우선 연습 문제를 풀어 본 후 확인해 보기를 권장한다.

In [ ]:
# 배치 학습 함수, 배치 검증 함수, 조기 종료를 모두 갖춘 완성본
train_loader, valid_loader, test_loader = mnist_loaders()

model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS, PATIENCE = 20, 3
best_loss, wait, best_state, best_epoch = float('inf'), 0, None, 0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, valid_loader, criterion)
    print(f'{epoch:2d}/{EPOCHS} 훈련 {tr_loss:.4f} / 검증 {va_loss:.4f} ({va_acc:.2f}%)')
    if va_loss < best_loss:
        best_loss, wait, best_epoch = va_loss, 0, epoch
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'조기 종료 - {PATIENCE}회 연속 개선 없음')
            break

model.load_state_dict(best_state)
te_loss, te_acc = run_epoch(model, test_loader, criterion)
print(f'\n최적 {best_epoch} 에포크 / 평가 정확도 {te_acc:.2f}%')

배치 학습 함수와 검증 함수는 **기울기 계산 여부**와 **모드 전환**만 다르므로 `run_epoch()` 하나로 합치고 `optimizer` 인자 유무로 구분했다. 조기 종료는 검증 손실이 개선될 때마다 파라미터를 저장해 두었다가, 마지막에 되돌리는 방식이다.

## 연습 4-6

과적합을 확인하기 위해서는 훈련 데이터셋과 검증, 평가 데이터셋을 반드시 분리해야 한다. 그렇다면 검증 데이터셋과 평가 데이터셋을 따로 분리하는 이유는 무엇인지 유추해 보자.

힌트: 검증 데이터셋이 학습 과정에서 어떤 결정(조기 종료 시점 등)에 사용되는지 떠올려 보자.

### 풀이

검증 데이터셋은 **학습 과정의 의사 결정**에 쓰인다. 조기 종료 시점을 정하고, 학습률·은닉층 크기 같은 하이퍼파라미터를 고르고, 여러 모델 중 하나를 선택하는 데 사용된다.

그 결정을 반복하다 보면 모델이 **검증 데이터에 간접적으로 맞춰진다**. 검증 손실이 가장 낮은 설정을 고르는 행위 자체가 검증 데이터에 대한 최적화이기 때문이다. 그래서 검증 정확도는 실제 성능보다 높게 나오는 경향이 있다(검증 데이터 과적합).

평가 데이터셋은 이 모든 결정이 끝난 뒤 **딱 한 번만** 사용해, 한 번도 관여하지 않은 데이터에서의 성능을 측정한다. 그래야 실제 서비스에서 마주칠 처음 보는 데이터에 대한 성능을 정직하게 추정할 수 있다.

정리하면 **검증 = 모델을 고르는 자, 평가 = 최종 성적을 매기는 자**이며, 두 역할을 한 데이터가 겸하면 성적이 부풀려진다.

## 연습 4-7

[도전 문제] 훈련, 검증, 평가 용도로 데이터를 구분하기에는 데이터의 양이 적은 경우에 다음과 같은 방법을 사용할 수 있다.

전체 데이터셋의 일부를 평가 데이터셋으로 구분한 후, 잠시 치워 둔다.

남은 데이터셋을 사용해 모델을 K번 반복 학습한다. 이를 위해 남은 데이터셋을 K개의 부분 데이터셋으로 나누고, 매번 학습할 때마다 K-1개의 부분 데이터셋을 훈련 데이터셋으로, 1개의 부분 데이터셋을 검증 데이터셋으로 사용한다. 이때 검증 데이터셋은 겹치지 않도록 돌려 가며 사용한다.

K번의 학습으로 계산한 최종 검증 오차의 평균을 일반화 성능의 지표로 사용한다.

처음 치워 두었던 평가 데이터셋으로 모델의 성능을 측정한다. K개의 모델 중 검증 성능이 가장 좋은 모델을 사용하거나 전체 데이터로 다시 학습한 모델을 사용한다.

이런 검증 방법을 교차 검증cross-validation이라고 하며, 위에서 소개한 방법은 대표적인 교차 검증법인 K-겹 교차 검증법K-fold cross-validation이다. 교차 검증은 데이터가 부족할 때뿐 아니라, 검증 데이터를 계속 바꿔 사용해 검증 손실의 신뢰성을 높이거나 특정 검증 데이터에 대한 과적합을 막는 용도로도 사용된다.

900개의 샘플로 구성된 회오리 모양 데이터(data/ch3_spiral_data.csv 파일)를 사용해 K=5인 K-겹 교차 검증법으로 모델을 학습하고 모델의 최종 분류 성능을 계산해 보자. random_split() 함수와 반대로 데이터셋을 합쳐야 하는 경우 torch.utils.data.ConcatDataset 클래스를 다음과 같이 사용하면 된다.

*코드 4-18 데이터셋을 합치는 방법*

```python
from torch.utils.data import ConcatDataset
# 합치는 데이터셋은 같은 구조의 샘플을 반환하는 Dataset 객체여야 모델 학습에 사용할 수 있음
combined_dataset = ConcatDataset([train_set, valid_set, test_set])
print(f'합친 데이터셋의 샘플 수: {len(combined_dataset)}')          # 출력값: 900
```

In [ ]:
# K-폴드 교차 검증 (K=5)
X, Y = load_spiral()
g = torch.Generator().manual_seed(SEED)
idx = torch.randperm(len(X), generator=g)
n_test = int(len(X) * 0.2)
test_idx, rest_idx = idx[:n_test], idx[n_test:]          # 평가용은 치워 둔다.
Xte, Yte = X[test_idx], Y[test_idx]

K = 5
folds = [rest_idx[i::K] for i in range(K)]               # K개 부분 데이터셋
n_class = int(Y.max()) + 1
fold_scores, states = [], []

for k in range(K):
    va_idx = folds[k]
    tr_idx = torch.cat([folds[j] for j in range(K) if j != k])
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                          nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, n_class))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(1500):
        loss = criterion(model(X[tr_idx]), Y[tr_idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    with torch.no_grad():
        acc = ((model(X[va_idx]).argmax(dim=1) == Y[va_idx]).float().mean() * 100).item()
    fold_scores.append(acc)
    states.append({kk: v.clone() for kk, v in model.state_dict().items()})
    print(f'{k + 1}번째 폴드 검증 정확도: {acc:.2f}%')

mean = sum(fold_scores) / K
var = sum((s - mean) ** 2 for s in fold_scores) / K
print(f'\n평균 {mean:.2f}% (표준편차 {var ** 0.5:.2f})')

# 가장 좋은 폴드의 모델로 평가 데이터셋 성능 측정
best_k = max(range(K), key=lambda i: fold_scores[i])
model.load_state_dict(states[best_k])
with torch.no_grad():
    print(f'최적 폴드({best_k + 1}) 모델의 평가 정확도: '
          f'{((model(Xte).argmax(dim=1) == Yte).float().mean() * 100).item():.2f}%')

K-폴드 교차 검증은 모든 데이터가 **한 번씩 검증 역할**을 맡으므로, 데이터가 적을 때 검증 결과가 우연에 좌우되는 문제를 줄여 준다. 폴드별 점수의 **표준편차**를 함께 보면 성능이 얼마나 안정적인지도 알 수 있다.

대신 K번 학습해야 하므로 비용이 K배로 늘어, 데이터와 모델이 큰 딥러닝에서는 상대적으로 덜 쓰인다.